In [1]:
# Imports and Setup
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Dataset
texts = ["i love this movie", "this film is amazing", "what a wonderful story"]
labels = [1, 1, 1]

# 2. Build Vocabulary
def build_vocab(texts):
    vocab = {"<pad>": 0}

    for text in texts:
        for word in text.split():
            if word not in vocab:
                vocab[word] = len(vocab)

    return vocab

vocab = build_vocab(texts)

In [3]:
# ---------------------------------
# 3. Encoding (TODO 1)
# TODO 1:

def encode_text(text, vocab):
    return [vocab.get(word, 0) for word in text.split()]

In [4]:
# ---------------------------------
# 4. Padding (TODO 2)
# TODO 2:

def pad_sequence(seq, max_len, pad_value=0):

    if len(seq) < max_len:
        seq = seq + [pad_value] * (max_len - len(seq))
    else:
        seq = seq[:max_len]

    return seq

In [5]:
# ---------------------------------
# Dataset Preparation

MAX_LEN = 10

encoded = [encode_text(t, vocab) for t in texts]
padded = [pad_sequence(e, MAX_LEN) for e in encoded]

X = torch.tensor(padded, dtype=torch.long)
y = torch.tensor(labels, dtype=torch.long)

dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

In [6]:
# ---------------------------------
# 6. Transformer Encoder Model (TODO 3, 4, 5)

class TransformerSentimentClassifier(nn.Module):

    def __init__(self, vocab_size, input_size, num_heads,
                 num_layers, num_classes, max_len, pad_idx):

        super().__init__()

        # TODO 3.1:

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=input_size,
            padding_idx=pad_idx
        )

        # TODO 3.2:

        self.pos_embedding = nn.Embedding(
            num_embeddings=max_len,
            embedding_dim=input_size
        )

        # TODO 4:

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=input_size,
            nhead=num_heads,
            dim_feedforward=128,
            dropout=0.1,
            batch_first=True
        )

        # TODO 5:

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.fc = nn.Linear(input_size, num_classes)

    def forward(self, x):

        batch_size, seq_len = x.shape

        x = self.embedding(x)

        positions = torch.arange(
            0, seq_len,
            device=x.device
        ).unsqueeze(0).expand(batch_size, seq_len)

        pos = self.pos_embedding(positions)

        x = x + pos

        out = self.transformer_encoder(x)

        out = out[:, -1, :]

        out = self.fc(out)

        return out


In [7]:
# ---------------------------------
# Model Initialization

model = TransformerSentimentClassifier(
    vocab_size=len(vocab),
    input_size=32,
    num_heads=2,
    num_layers=2,
    num_classes=2,
    max_len=MAX_LEN,
    pad_idx=0
).to(device)

In [8]:
# ---------------------------------
# 7. Inference Function (TODO 6)

# TODO 6:

def predict_sentiment(model, text, vocab, max_len, device):

    model.eval()

    # Encode text
    encoded = encode_text(text, vocab)

    # Pad sequence
    padded = pad_sequence(encoded, max_len)

    # Convert to tensor
    tensor = torch.tensor([padded], dtype=torch.long).to(device)

    # Prediction
    with torch.no_grad():
        output = model(tensor)
        prediction = torch.argmax(output, dim=1).item()

    return "Positive" if prediction == 1 else "Negative"

# ---------------------------------
# Example

result = predict_sentiment(
    model,
    "this movie is wonderful",
    vocab,
    MAX_LEN,
    device
)

print(result)

Positive
